# Partie 2 - Neural Network : Auto-encodeur MNIST

Ce notebook reprend l'activité de la vidéo : passer d'un modèle discriminatif MNIST à un premier modèle génératif simple, l'auto-encodeur.

## 1. Préparation

Le notebook original mentionné dans la vidéo est à récupérer sur Classroom, dans l'activité pratique numéro 2. Cette version locale contient les exercices nécessaires : chargement MNIST, classifieur, auto-encodeur, reconstructions et espace latent.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Images d'entrainement :", len(train_dataset))
print("Images de test        :", len(test_dataset))

In [ ]:
images, labels = next(iter(train_loader))

print("Batch images :", images.shape)   # [batch, canal, hauteur, largeur]
print("Batch labels :", labels.shape)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(images[i, 0], cmap="gray")
    ax.set_title(f"Label : {labels[i].item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Modèle discriminatif : classifieur MNIST

Avant de construire le modèle génératif, on entraîne un réseau de neurones classique qui prédit le chiffre de l'image.

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


classifier = Classifier().to(device)
print(classifier)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    classifier.train()
    total_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = classifier(x_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_batch.size(0)

    average_loss = total_loss / len(train_dataset)
    print(f"Epoque {epoch + 1}/{epochs} | Loss : {average_loss:.4f}")

In [ ]:
classifier.eval()
correct = 0
total = 0

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = classifier(x_batch)
        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total
print(f"Accuracy test : {accuracy:.2%}")

## 3. Modèle génératif : auto-encodeur simple

L'encodeur transforme une image `28 x 28` en vecteur latent `z`. Le décodeur reconstruit une image à partir de ce vecteur.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        x_hat = self.decoder(z)
        return x_hat.view(-1, 1, 28, 28)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat


autoencoder = AutoEncoder(latent_dim=2).to(device)
print(autoencoder)

In [ ]:
criterion_ae = nn.MSELoss()
optimizer_ae = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

epochs = 5
ae_losses = []

for epoch in range(epochs):
    autoencoder.train()
    total_loss = 0.0

    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)

        x_hat = autoencoder(x_batch)
        loss = criterion_ae(x_hat, x_batch)

        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()

        total_loss += loss.item() * x_batch.size(0)

    average_loss = total_loss / len(train_dataset)
    ae_losses.append(average_loss)
    print(f"Epoque {epoch + 1}/{epochs} | Reconstruction loss : {average_loss:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ae_losses)
plt.xlabel("Epoque")
plt.ylabel("MSE")
plt.title("Evolution de la perte de reconstruction")
plt.grid(True)
plt.show()

In [ ]:
autoencoder.eval()

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    reconstructions = autoencoder(images)

n = 10
fig, axes = plt.subplots(2, n, figsize=(15, 4))

for i in range(n):
    axes[0, i].imshow(images[i, 0].cpu(), cmap="gray")
    axes[0, i].set_title(f"Original\n{labels[i].item()}")
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructions[i, 0].cpu(), cmap="gray")
    axes[1, i].set_title("Reconstruit")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
autoencoder.eval()

with torch.no_grad():
    z_random = torch.randn(16, 2, device=device)
    generated = autoencoder.decode(z_random)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(generated[i, 0].cpu(), cmap="gray")
    ax.axis("off")

plt.suptitle("Images generees depuis l'espace latent")
plt.tight_layout()
plt.show()

In [ ]:
autoencoder.eval()

z_values = []
y_values = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        z = autoencoder.encode(x_batch)
        z_values.append(z.cpu())
        y_values.append(y_batch)

z_values = torch.cat(z_values)
y_values = torch.cat(y_values)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    z_values[:, 0],
    z_values[:, 1],
    c=y_values,
    cmap="tab10",
    s=8,
    alpha=0.7,
)
plt.colorbar(scatter, ticks=range(10), label="Chiffre")
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Espace latent de l'auto-encodeur")
plt.grid(True)
plt.show()